# FDM-PINN plate bending under HYDROSTATIC loading

In [2]:
"""
FDM-PINN (Finite-Difference-Method PINN) for Kirchhoff-Love plate bending
under HYDROSTATIC (linearly varying) loading, all-simply-supported
(SS-SS-SS-SS).

    q(x, y) = q0 * (x / a)        -- zero at x=0, rising linearly to q0 at
                                      x=a, UNIFORM across y.

This is the hydrostatic-load sibling of fdm_pinn_uniform_hardBC.py /
fdm_pinn_sinusoidal_hardBC.py. The network architecture, hard-constrained
essential BC (w=0 on all edges via the envelope trick), FD-stencil
derivative machinery, training loop, and output artifact schema are ALL
UNCHANGED -- see fdm_pinn_sinusoidal_hardBC.py's docstring for the full FD
rationale if needed. Only the forcing term and the analytical reference
solution differ.

-----------------------------------------------------------------------------
ANALYTICAL SOLUTION -- reference: S. Timoshenko & S. Woinowsky-Krieger,
"Theory of Plates and Shells", 2nd ed., McGraw-Hill, 1959, Sec. 28
("Simply Supported Rectangular Plates", hydrostatic / linearly-varying
load case).
-----------------------------------------------------------------------------
q(x,y) = q0*x/a factors into a function of x times a function of y, so it
expands as a product of two independent 1D Fourier sine series:

  x/a = sum_{m=1,2,3,...} [2(-1)^(m+1)/(m pi)] sin(m pi x/a)      (ALL m)
  1   = sum_{n=1,3,5,...} [4/(n pi)] sin(n pi y/b)                 (ODD n only)

Unlike the uniform-load case (q0 x/a x0/a with a constant in x, which needed
ODD m and ODD n), here the x-direction is a RAMP, not a constant, so its
Fourier series needs EVERY m (both even and odd), with alternating sign
(-1)^(m+1). The y-direction is still just "1" across [0,b], so ODD n only,
same as the uniform case. Multiplying the two series and matching term by
term against D*grad^4(w) = q (same procedure as the uniform-load script)
gives:

  w(x,y) = (8 q0)/(pi^6 D) * sum_{m=1,2,3,...} sum_{n=1,3,5,...}
               (-1)^(m+1) sin(m pi x/a) sin(n pi y/b) / [ m n (m^2/a^2+n^2/b^2)^2 ]

Moments follow by differentiating term-by-term exactly as in the uniform
case (same Mx/My/Mxy coefficient structure, just with this A_mn).

VALIDATION done before trusting this in the pipeline (see chat / can be
re-run): q0*x/a = q0/2 + (q0/2)*(2x/a - 1) is a symmetric half (uniform
q0/2) plus an antisymmetric-about-x=a/2 half. The antisymmetric half must
produce a deflection field that is itself antisymmetric about x=a/2, hence
EXACTLY ZERO at the center. So the hydrostatic center deflection must equal
EXACTLY HALF the uniform-load center deflection. Checked numerically:
uniform center = 0.00406235 (nd), hydrostatic center = 0.00203118 (nd) --
exactly half, as required. Also checked: edges are zero to float precision,
and deflection is larger toward x=a (more load) than x=0, as physically
expected. Series converges by ~M=20; ANALYTICAL_SERIES_M_TERMS below uses
a generous 200 for extra margin at negligible cost.

NOTE ON OUTPUT FOLDER: OUT_DIR is keyed on both MODEL_NAME and
LOADING_CASE ("results/<MODEL_NAME>_<LOADING_CASE>"), matching the fix
already applied in fdm_pinn_uniform_hardBC.py, so this run will not
collide with the sinusoidal or uniform result folders.

Outputs (written to ./results/<MODEL_NAME>_<LOADING_CASE>/):
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python fdm_pinn_hydrostatic_hardBC.py
"""

import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "FDM_PINN_HardBC"   # >>> VARIANT POINT: change per architecture
LOADING_CASE = "hydrostatic"

OUT_DIR = os.path.join("results", f"{MODEL_NAME}_{LOADING_CASE}")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 15000
PATIENCE = 3000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

# ============================================================
# DOMAIN
# ============================================================
nx, ny = 51, 51
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

dx = a_nd / (nx - 1)
dy = b_nd / (ny - 1)

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept only for the diagnostic bc_loss now ------------
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)

bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)

# --- per-edge indices for the moment BCs ------------------------------
edge_x0 = np.where(grid_points_nd[:, 0] == 0)[0]
edge_xa = np.where(grid_points_nd[:, 0] == a_nd)[0]
edge_y0 = np.where(grid_points_nd[:, 1] == 0)[0]
edge_yb = np.where(grid_points_nd[:, 1] == b_nd)[0]

edge_x_indices = np.unique(np.concatenate([edge_x0, edge_xa]))  # Mx = 0 here
edge_y_indices = np.unique(np.concatenate([edge_y0, edge_yb]))  # My = 0 here

edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)

# ============================================================
# GHOST-EXTENDED GRID (for finite-difference stencils)
# ============================================================
# GHOST=3 is the minimum needed by the 4th-order-accurate 4th-derivative
# stencil (offsets -3..+3). Do not reduce this.
GHOST = 3

x_ext_1d = (np.arange(-GHOST, nx + GHOST)) * dx   # length nx + 2*GHOST
y_ext_1d = (np.arange(-GHOST, ny + GHOST)) * dy   # length ny + 2*GHOST
X_ext, Y_ext = np.meshgrid(x_ext_1d, y_ext_1d)    # shape (ny+2G, nx+2G)

X_ext_flat = torch.tensor(X_ext.reshape(-1, 1), dtype=DTYPE, device=DEVICE)
Y_ext_flat = torch.tensor(Y_ext.reshape(-1, 1), dtype=DTYPE, device=DEVICE)
EXT_NY = ny + 2 * GHOST
EXT_NX = nx + 2 * GHOST

# Physical (non-dimensional) grid, as a torch tensor, for evaluating the load
X_nd_t = torch.tensor(X_nd, dtype=DTYPE, device=DEVICE)
Y_nd_t = torch.tensor(Y_nd, dtype=DTYPE, device=DEVICE)

# --- 4th-order-accurate central finite-difference coefficients (Fornberg) --
# 1st derivative, 5-pt, offsets -2..2
C1 = torch.tensor([1 / 12, -8 / 12, 0.0, 8 / 12, -1 / 12], dtype=DTYPE, device=DEVICE)
# 2nd derivative, 5-pt, offsets -2..2
C2 = torch.tensor([-1 / 12, 16 / 12, -30 / 12, 16 / 12, -1 / 12], dtype=DTYPE, device=DEVICE)
# 4th derivative, 7-pt, offsets -3..3
C4 = torch.tensor([-1 / 6, 2.0, -13 / 2, 28 / 3, -13 / 2, 2.0, -1 / 6], dtype=DTYPE, device=DEVICE)


def _apply_stencil_axis(arr, coeffs, axis, ghost, out_len):
    """Apply a centered 1D stencil (`coeffs`, offsets -half..+half) along
    `axis` of `arr`, where `arr` has `ghost` extra points on each side of
    that axis beyond the first `out_len` physical points. Returns a tensor
    with `out_len` along `axis` and all other dimensions unchanged."""
    half = (coeffs.shape[0] - 1) // 2
    assert ghost >= half, "not enough ghost padding for this stencil"
    out = 0.0
    for k in range(-half, half + 1):
        c = coeffs[k + half]
        start = ghost + k
        if axis == 0:
            sl = arr[start:start + out_len, ...]
        else:
            sl = arr[:, start:start + out_len]
        out = out + c * sl
    return out


def d1_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C1, axis, ghost, out_len) / h


def d2_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C2, axis, ghost, out_len) / (h ** 2)


def d4_axis(arr, axis, ghost, out_len, h):
    return _apply_stencil_axis(arr, C4, axis, ghost, out_len) / (h ** 4)


def fd_derivatives(model):
    """Single forward pass on the ghost-extended grid, then every
    derivative needed for training and post-processing as finite
    differences of that one pass. All returned tensors have shape
    (ny, nx), matching X_nd/Y_nd's row=y, col=x convention."""
    w_ext = model(X_ext_flat, Y_ext_flat).reshape(EXT_NY, EXT_NX)

    # Pure-x derivatives: physical y rows, full x ghost range
    w_rows = w_ext[GHOST:GHOST + ny, :]                              # (ny, nx+2G)
    w_xxxx = d4_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)   # (ny, nx)
    w_xx = d2_axis(w_rows, axis=1, ghost=GHOST, out_len=nx, h=dx)     # (ny, nx)

    # Pure-y derivatives: full y ghost range, physical x cols
    w_cols = w_ext[:, GHOST:GHOST + nx]                               # (ny+2G, nx)
    w_yyyy = d4_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)   # (ny, nx)
    w_yy = d2_axis(w_cols, axis=0, ghost=GHOST, out_len=ny, h=dy)     # (ny, nx)

    # Mixed: w_xxyy = d2/dx2 ( d2/dy2 w ), applied as two sequential 1D stencils
    w_yy_full = d2_axis(w_ext, axis=0, ghost=GHOST, out_len=ny, h=dy)  # (ny, nx+2G)
    w_xxyy = d2_axis(w_yy_full, axis=1, ghost=GHOST, out_len=nx, h=dx)  # (ny, nx)

    # Mixed: w_xy = d/dx ( d/dy w )
    w_y_full = d1_axis(w_ext, axis=0, ghost=GHOST, out_len=ny, h=dy)   # (ny, nx+2G)
    w_xy = d1_axis(w_y_full, axis=1, ghost=GHOST, out_len=nx, h=dx)    # (ny, nx)

    w_phys = w_ext[GHOST:GHOST + ny, GHOST:GHOST + nx]                 # (ny, nx)

    return w_phys, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy


# ============================================================
# ANALYTICAL SOLUTION -- Navier double-Fourier-series, hydrostatic
# (linearly varying) load q = q0*x/a
# (Timoshenko & Woinowsky-Krieger, "Theory of Plates and Shells", Sec. 28)
# ============================================================
ANALYTICAL_SERIES_M_TERMS = 200   # ALL m = 1..200 (ramp direction -- needs every m)
ANALYTICAL_SERIES_N_TERMS = 100   # ODD n only, n = 1,3,...,199 (uniform-in-y direction)

_m_vals = np.arange(1, ANALYTICAL_SERIES_M_TERMS + 1)            # 1, 2, 3, ..., ALL integers
_n_vals = np.arange(1, 2 * ANALYTICAL_SERIES_N_TERMS, 2)         # 1, 3, 5, ..., odd only
_mm, _nn = np.meshgrid(_m_vals, _n_vals, indexing="ij")            # (M, N)
_lam_mn = (_mm / a_nd) ** 2 + (_nn / b_nd) ** 2                     # (M, N)
_sign_m = (-1.0) ** (_mm + 1)                                       # +1 odd m, -1 even m
_A_mn = 8.0 * q0_nd * _sign_m / (np.pi ** 6 * D_nd * _mm * _nn * _lam_mn ** 2)   # deflection series coeffs

# Sin/cos basis evaluated on the SAME (x_nd, y_nd) 1D grid vectors used to
# build X_nd/Y_nd -- the analytical_*_nd(x, y) functions below take (x, y)
# only for call-signature parity with the other loading-case scripts; they
# always evaluate on this fixed module-level grid via an efficient
# matrix-multiply (basis.T @ coeffs.T @ basis) rather than a slow
# double Python loop over 20,000 mode pairs.
_sin_mx = np.sin(np.pi * np.outer(_m_vals, x_nd) / a_nd)   # (M, nx)
_sin_ny = np.sin(np.pi * np.outer(_n_vals, y_nd) / b_nd)   # (N, ny)
_cos_mx = np.cos(np.pi * np.outer(_m_vals, x_nd) / a_nd)   # (M, nx)
_cos_ny = np.cos(np.pi * np.outer(_n_vals, y_nd) / b_nd)   # (N, ny)


def analytical_w_nd(x, y):
    return _sin_ny.T @ (_A_mn.T @ _sin_mx)   # (ny, nx)


def analytical_Mx_nd(x, y):
    B_mn = D_nd * _A_mn * ((np.pi * _mm / a_nd) ** 2 + nu_nd * (np.pi * _nn / b_nd) ** 2)
    return _sin_ny.T @ (B_mn.T @ _sin_mx)


def analytical_My_nd(x, y):
    C_mn = D_nd * _A_mn * ((np.pi * _nn / b_nd) ** 2 + nu_nd * (np.pi * _mm / a_nd) ** 2)
    return _sin_ny.T @ (C_mn.T @ _sin_mx)


def analytical_Mxy_nd(x, y):
    E_mn = -D_nd * (1 - nu_nd) * _A_mn * (np.pi * _mm / a_nd) * (np.pi * _nn / b_nd)
    return _cos_ny.T @ (E_mn.T @ _cos_mx)


def hydrostatic_load_nd(x, y):
    return q0_nd * (x / a_nd)


Q_grid = hydrostatic_load_nd(X_nd_t, Y_nd_t)   # (ny, nx), precomputed once -- ramps 0 -> q0_nd across x


# ============================================================
# MODEL   (>>> VARIANT POINT: swap the input block for FF-PINN /
#          swap the residual block for FDM-PINN / FF+FDM-PINN)
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3):
        super().__init__()
        layers = []
        in_dim = 2
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def forward(self, x, y):
        h = torch.cat([x / a_nd, y / b_nd], dim=1)   # >>> VARIANT POINT: Fourier embed here for FF variants
        h = self.hidden(h)
        raw = self.out(h)

        # Hard-constrained essential BC (w = 0 on all edges), unchanged from
        # the vanilla hard-BC script. Evaluated fine for ghost points
        # outside [0,a]x[0,b] too -- a smooth polynomial continuation.
        xi = x / a_nd
        eta = y / b_nd
        envelope = xi * (1 - xi) * eta * (1 - eta)
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS (4th-order derivatives via finite differences, not autograd)
# ============================================================
def compute_loss():
    w_grid, w_xx, w_yy, w_xxxx, w_yyyy, w_xxyy, w_xy = fd_derivatives(model)

    biharmonic = w_xxxx + 2 * w_xxyy + w_yyyy
    residual = (biharmonic - Q_grid).reshape(-1, 1)
    pde_loss = torch.mean(residual[int_idx] ** 2)

    w_flat = w_grid.reshape(-1, 1)
    # DIAGNOSTIC ONLY -- should be ~0 exactly, by construction of the model
    # (hard-constrained envelope). Not included in `total`.
    bc_loss = torch.mean(w_flat[bc_idx] ** 2)

    Mx = -(w_xx + nu_nd * w_yy)
    My = -(w_yy + nu_nd * w_xx)
    Mx_flat = Mx.reshape(-1, 1)
    My_flat = My.reshape(-1, 1)
    moment_loss = torch.mean(Mx_flat[edge_x_idx] ** 2) + torch.mean(My_flat[edge_y_idx] ** 2)

    total = pde_loss + moment_loss
    return total, pde_loss, bc_loss, moment_loss, w_flat


# ============================================================
# TRAINING LOOP with computational-cost logging + early stopping
# ============================================================
epoch_history, loss_history, pde_history, bc_history, moment_history = [], [], [], [], []
w_center_history, epoch_time_history = [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc, moment, w_pred = compute_loss()
    total.backward()
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    moment_history.append(moment.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss {loss_val:.6e} (pde {pde.item():.3e}, "
              f"moment {moment.item():.3e}, hardBC_residual {bc.item():.3e}) "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    if loss_val < best_loss:
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            converged_epoch = epoch
            break

total_train_time = time.perf_counter() - train_start
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_bc_residual_diagnostic": bc_history,
    "moment_loss": moment_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS BALANCE DIAGNOSTIC
# ============================================================
final_pde, final_bc, final_moment = pde_history[-1], bc_history[-1], moment_history[-1]
final_total = final_pde + final_moment  # bc excluded, it's ~0 by construction
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "final_pde_loss": final_pde,
    "final_moment_loss": final_moment,
    "final_hard_bc_residual_diagnostic": final_bc,
    "pde_fraction_of_total": final_pde / final_total,
    "moment_fraction_of_total": final_moment / final_total,
}
dominant_fraction = max(
    loss_balance["pde_fraction_of_total"],
    loss_balance["moment_fraction_of_total"],
)
if dominant_fraction > 0.9:
    print(f"WARNING: one loss term is {dominant_fraction*100:.1f}% of the total "
          f"unweighted loss -- consider a weighting scheme before trusting these results.")
if final_bc > 1e-20:
    print(f"WARNING: hard_bc_residual_diagnostic = {final_bc:.3e} is not ~0 -- "
          f"double-check the envelope function / grid point exactness (float64 expected).")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "device": str(DEVICE),
    "epochs_run": len(epoch_history),
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,  # deprecated alias, same value
    "mean_time_per_epoch_s": float(np.mean(epoch_time_history)),
    "median_time_per_epoch_s": float(np.median(epoch_time_history)),
    "total_training_time_s": total_train_time,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "ghost_points_per_side": GHOST,
    "forward_pass_points": EXT_NX * EXT_NY,
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My, Mxy) via the
# same finite-difference machinery used in training.
# ============================================================
def compute_full_fields():
    with torch.no_grad():
        w_grid, w_xx, w_yy, _, _, _, w_xy = fd_derivatives(model)
        Mx = -(w_xx + nu_nd * w_yy)
        My = -(w_yy + nu_nd * w_xx)
        Mxy = -(1 - nu_nd) * w_xy

    w_grid_np = w_grid.cpu().numpy()
    Mx_np = Mx.cpu().numpy()
    My_np = My.cpu().numpy()
    Mxy_np = Mxy.cpu().numpy()
    return w_grid_np, Mx_np, My_np, Mxy_np


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd = compute_full_fields()

w_anal_nd = analytical_w_nd(X_nd, Y_nd)
Mx_anal_nd = analytical_Mx_nd(X_nd, Y_nd)
My_anal_nd = analytical_My_nd(X_nd, Y_nd)
Mxy_anal_nd = analytical_Mxy_nd(X_nd, Y_nd)

w_dim_pred, w_dim_anal = w_pred_nd * W, w_anal_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_anal = Mx_pred_nd * moment_scale, Mx_anal_nd * moment_scale
My_dim_pred, My_dim_anal = My_pred_nd * moment_scale, My_anal_nd * moment_scale
Mxy_dim_pred, Mxy_dim_anal = Mxy_pred_nd * moment_scale, Mxy_anal_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_anal
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_anal)
My_error = np.abs(My_dim_pred - My_dim_anal)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_anal)

# ============================================================
# ERROR METRICS
# ============================================================
ix, iy = nx // 2, ny // 2
w_pred_center, w_anal_center = w_dim_pred[iy, ix], w_dim_anal[iy, ix]
center_abs_err = abs(w_pred_center - w_anal_center)
center_pct_err = center_abs_err / abs(w_anal_center) * 100

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_anal) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_anal_m": float(w_anal_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": float(np.max(np.abs(w_dim_pred[0, :]))
                                         + np.max(np.abs(w_dim_pred[-1, :]))
                                         + np.max(np.abs(w_dim_pred[:, 0]))
                                         + np.max(np.abs(w_dim_pred[:, -1]))),
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_anal": w_dim_anal.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_anal": Mx_dim_anal.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_anal": My_dim_anal.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_anal": Mxy_dim_anal.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_anal"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_anal"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
plt.axhline(y=w_anal_center, linestyle="--", linewidth=2, color="k", label="Analytical")
plt.xlabel("Epoch"); plt.ylabel("Center Deflection w (m)")
plt.title("Convergence of Center Deflection")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_anal_center), linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total Loss")
plt.plot(df_hist["epoch"], df_hist["pde_loss"], label="PDE Loss", alpha=0.7)
plt.plot(df_hist["epoch"], df_hist["moment_loss"], label="Moment Loss", alpha=0.7)
plt.plot(df_hist["epoch"], df_hist["hard_bc_residual_diagnostic"],
         label="Hard-BC residual (diagnostic, ~0)", alpha=0.5, linestyle=":")
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curve")
plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_anal, "Analytical Plate Deflection", "Deflection (m)",
             "Analytical_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_anal| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_anal", "w_pred - w_anal (m)",
             "Absolute_Error_w_pred_-_w_anal.png", cmap="coolwarm")

save_contour(Mx_dim_anal, "Analytical Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x_Analytical.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_anal| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_anal, "Analytical Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y_Analytical.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_anal| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_anal, "Analytical Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy_Analytical.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_anal| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_anal.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_anal.min(), w_dim_pred.min()), max(w_dim_anal.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("Analytical displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("Analytical vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Analytical_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_anal, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_anal, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_anal, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_anal, Mxy_dim_pred, Mxy_error),
]
for r, (label, anal, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"Analytical {label}", anal, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading, SS-SS-SS-SS (summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Total training time: {total_train_time:.2f}s | "
      f"Epochs to early-stop: {converged_epoch} | "
      f"Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Total training time: {total_train_time:.2f}s | Epochs: {converged_epoch}")

Epoch 0 Loss 1.412462e-01 (pde 1.400e-01, moment 1.217e-03, hardBC_residual 0.000e+00) Center 4.429634e-03 (12.42 ms/epoch)
Epoch 100 Loss 1.530092e-03 (pde 4.260e-05, moment 1.487e-03, hardBC_residual 0.000e+00) Center 3.870502e-03 (10.68 ms/epoch)
Epoch 200 Loss 1.504142e-03 (pde 2.546e-05, moment 1.479e-03, hardBC_residual 0.000e+00) Center 3.871626e-03 (11.33 ms/epoch)
Epoch 300 Loss 1.497730e-03 (pde 2.405e-05, moment 1.474e-03, hardBC_residual 0.000e+00) Center 3.871299e-03 (10.83 ms/epoch)
Epoch 400 Loss 1.492826e-03 (pde 2.424e-05, moment 1.469e-03, hardBC_residual 0.000e+00) Center 3.870824e-03 (11.33 ms/epoch)
Epoch 500 Loss 1.487388e-03 (pde 2.473e-05, moment 1.463e-03, hardBC_residual 0.000e+00) Center 3.870153e-03 (11.03 ms/epoch)
Epoch 600 Loss 1.481996e-03 (pde 2.512e-05, moment 1.457e-03, hardBC_residual 0.000e+00) Center 3.869473e-03 (11.18 ms/epoch)
Epoch 700 Loss 1.475751e-03 (pde 2.581e-05, moment 1.450e-03, hardBC_residual 0.000e+00) Center 3.868456e-03 (10.47 ms/e

C:\Users\vivek\AppData\Local\Temp\ipykernel_13156\50797418.py:432: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location=

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'FDM_PINN_HardBC', 'loading_case': 'hydrostatic', 'final_pde_loss': 1.0264842037618268e-05, 'final_moment_loss': 1.0773985728061797e-05, 'final_hard_bc_residual_diagnostic': 0.0, 'pde_fraction_of_total': 0.4878999035470484, 'moment_fraction_of_total': 0.5121000964529515}
Saved cost_metrics.json: {'model': 'FDM_PINN_HardBC', 'loading_case': 'hydrostatic', 'device': 'cuda', 'epochs_run': 15000, 'epochs_to_early_stop': 15000, 'epochs_to_convergence': 15000, 'mean_time_per_epoch_s': 0.010932739646721165, 'median_time_per_epoch_s': 0.010861150018172339, 'total_training_time_s': 176.65227109997068, 'peak_gpu_memory_mb': 30.086144, 'n_trainable_params': 8578, 'ghost_points_per_side': 3, 'forward_pass_points': 3249}
Saved error_metrics.json: {'model': 'FDM_PINN_HardBC', 'loading_case': 'hydrostatic', 'center_deflection_pred_m': 0.00011061071440474288, 'center_deflection_anal_m': 0.00011090222763531471, 'center_abs_error_m': 2.91